### New experiments using OSM. 
* We will pull out the relevant data in tags of osm
* Then we will filter to Exeter

In [ ]:
import pyrosm
import geopandas as gpd
import pandas as pd

PBF_FILE = r"osm_data\devon-260326.osm.pbf"

print("Reading OSM data...")
osm = pyrosm.OSM(PBF_FILE)

# Helper function to safely save a layer
def save_layer(gdf, name):
    if gdf is not None and len(gdf) > 0:
        print(f"{name}: {len(gdf):,} features")
        gdf.to_file(f"devon_{name}.gpkg", driver="GPKG")
        print(f"✓ Saved devon_{name}.gpkg")
    else:
        print(f"{name}: empty or not found")

# ── Buildings ─────────────────────────────────────────────
print("\nReading buildings...")
buildings = osm.get_buildings()
save_layer(buildings, "buildings")

# ── Network / Roads ───────────────────────────────────────
print("\nReading roads/network...")
network = osm.get_network(network_type="all")
save_layer(network, "network")

# ── Landuse ───────────────────────────────────────────────
print("\nReading landuse...")
landuse = osm.get_landuse()
save_layer(landuse, "landuse")

# ── POIs ──────────────────────────────────────────────────
print("\nReading POIs...")
pois = osm.get_pois()
save_layer(pois, "pois")

# ── Natural features ──────────────────────────────────────
print("\nReading natural features...")
natural = osm.get_natural()
save_layer(natural, "natural")

# ── Waterways ─────────────────────────────────────────────
print("\nReading waterways...")
waterways = osm.get_data_by_custom_criteria(custom_filter={"waterway": True})
save_layer(waterways, "waterways")

# ── Boundaries ────────────────────────────────────────────
print("\nReading boundaries...")
boundaries = osm.get_data_by_custom_criteria(custom_filter={"boundary": True})
save_layer(boundaries, "boundaries")

print("\nDone!")

### Post filtering of OSM data 
* Removing irrelevant columns
* Finding filter columns

#### Buildings

In [ ]:
import geopandas as gpd
gdf_polygon = gpd.read_file(r"osm_data\devon_buildings.gpkg")
gdf_polygon.columns

In [ ]:
recommended_columns = ['addr:city', 'addr:country', 'addr:housenumber', 'addr:housename',
       'addr:postcode', 'addr:place', 'addr:street','name',
       'opening_hours', 'website', 'building', 'amenity', 'building:flats', 'building:levels',
       'building:material', 'building:min_level', 'building:use','craft',
       'height', 'internet_access', 'landuse', 'levels', 'office', 'shop',
       'source','geometry']

In [ ]:
gdf_polygon.building.isna().any()


In [ ]:
invalid = gdf_polygon[~gdf_polygon.is_valid]
print(len(invalid))

In [ ]:
gdf_polygon.building.unique()

#### Landuse

In [ ]:
gdf_landuse = gpd.read_file(r"osm_data\devon_landuse.gpkg")
gdf_landuse.columns

In [ ]:
invalid = gdf_landuse[~gdf_landuse.is_valid]
print(len(invalid))

In [ ]:
gdf_landuse.crs

In [ ]:
gdf_landuse.head()

In [ ]:
gdf_landuse.landuse.unique()

In [ ]:
gdf_landuse.landuse.isna().any()

#### Natural

In [ ]:
gdf_natural = gpd.read_file(r"osm_data\devon_natural.gpkg")
gdf_natural.columns

In [ ]:
invalid = gdf_natural[~gdf_natural.is_valid]
print(len(invalid))

In [ ]:
gdf_natural.crs

In [ ]:
gdf_natural.natural.unique()

#### POI

In [ ]:
gdf_pois = gpd.read_file(r"osm_data\devon_pois.gpkg")
gdf_pois.columns

In [ ]:
invalid = gdf_pois[~gdf_pois.is_valid]
print(len(invalid))

In [ ]:
gdf_pois.crs

#### waterways

In [ ]:
gdf_waterways = gpd.read_file(r"osm_data\devon_waterways.gpkg")
gdf_waterways.columns

In [ ]:
invalid = gdf_waterways[~gdf_waterways.is_valid]
print(len(invalid))

In [ ]:
gdf_waterways.crs

In [ ]:
gdf_waterways.waterway.unique()

### Boundaries

In [ ]:
gdf_boundaries = gpd.read_file(r"osm_data\devon_boundaries.gpkg")
gdf_boundaries.columns

In [ ]:
invalid = gdf_boundaries[~gdf_boundaries.is_valid]
print(len(invalid))

In [ ]:
gdf_boundaries.crs

In [ ]:
gdf_boundaries.name.unique()

### Finally converting all to EPSG 27700

In [ ]:
import os
import geopandas as gpd
for filename in os.listdir(r"osm_data"):
    if filename.endswith(".gpkg"):
        gdf = gpd.read_file(os.path.join(r"osm_data", filename))
        gdf = gdf.to_crs(epsg=27700)
        gdf = gdf[gdf.is_valid]
        gdf.to_file(os.path.join(r"osm_data", filename), driver="GPKG")

In [ ]:
import geopandas as gpd
gdf_admin = gpd.read_file(r"osm_data\devon_boundaries.gpkg")
gdf_admin.name.value_counts()

In [ ]:
print(gdf_admin.name.value_counts())

In [ ]:
gdf_admin[gdf_admin.name == "Dorset"]

In [ ]:
import joblib
data = joblib.load(r"artifacts\waterway_river_clyst_exeter_search.pkl").data
data.columns

In [ ]:
data

### DOCUMENTATION : How to use this tool

In [2]:
import pandas as pd
import json
from utils.initialize_os_agents import OSAgentsInitializer
from utils.keys import set_api_keys
from utils.tools import human_send_message
set_api_keys()
import shutil
import joblib
import os

config = None
with open(r"agent_frameworks\agent_config_with_human_confirmation.json","rb") as file:
    config = json.load(file)

# Now that things are initialised
agent_archiecture = OSAgentsInitializer(config,"Question_0",diff_dir=None).initialize_all_agents()

human_send_message(message="Brilliant, can you also add the polygon of bickington to the map?",target_agent=[agent_archiecture["host_agent"]])

Openai and ngd key set successfully
Model chosen o3
Model chosen o3
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen gpt-4.1
Model chosen o3


["Your updated map is ready.\n\n• Artifact: bickington_buildings_buffer_map  \n  – An interactive map (HTML) displaying:  \n    1. The Bickington civil-parish boundary polygon (red outline).  \n    2. A 1 km buffer ring around that boundary (light blue shading).  \n    3. All 775 building footprints that fall inside the buffer (blue polygons).\n\nOpen bickington_buildings_buffer_map.html to explore the boundary, buffer, and buildings together.Addtionally some data artifacts have been generated with names  ['bickington_buildings_buffer_map'] and \n descriptions ['Interactive folium map with Bickington boundary, 1km buffer ring, and all buildings within 1km of the boundary.']",

In [ ]:
import pandas as pd
import json
from utils.initialize_os_agents import OSAgentsInitializer
from utils.keys import set_api_keys
from utils.tools import human_send_message
set_api_keys()
import shutil
import os
import geopandas as gpd
import numpy as np

gdf = gpd.read_file(r"osm_data\devon_boundaries.gpkg")
available_locations = gdf.name.unique()
available_locations = np.array([loc.lower() for loc in available_locations if loc is not None])
questions = pd.read_csv(r"evaluation\valid.csv")["query"].tolist()
questions = questions[2:]

question_number = 38
for question in questions[38:]:  # Start from the 33rd question (index 32)

    if "OSID" in question:
        continue
    

    location_randomly_sampled = np.random.choice(available_locations)

    question = question.replace("{input_location}",location_randomly_sampled)
    question = question.replace("{{location:Romsey}}",location_randomly_sampled)


    config = None
    with open(r"agent_frameworks\agent_config_with_human_confirmation.json","rb") as file:
        config = json.load(file)

    # Now that things are initialised
    agent_archiecture = OSAgentsInitializer(config,f"Question_{question_number}",diff_dir=None).initialize_all_agents()
    print(f"Starting Question {question_number} : {question}")

    human_send_message(message=question,target_agent=[agent_archiecture["host_agent"]])
    shutil.rmtree("artifacts")
    shutil.rmtree("message_store")
    os.makedirs("artifacts", exist_ok=True)
    os.makedirs("message_store", exist_ok=True)
    print(f"Completed Question {question_number}")
    print("-"*50)
    question_number += 1